# A6 — cohort and patient payloads

Split so the control board can switch *k* without refetching the patient.
Encoder identity is required. `assert_safe` runs on every generated string.


In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
for cand in [cwd, *cwd.parents]:
    if (cand / "src" / "gate.py").is_file():
        sys.path.insert(0, str(cand / "src"))
        break
    nested = cand / "v2"
    if (nested / "src" / "gate.py").is_file():
        sys.path.insert(0, str(nested / "src"))
        break

from paths import ensure_src_on_path, resolve_v2_root
from gate import gate as _gate_impl
from safety import assert_safe

V2_ROOT = resolve_v2_root()
ensure_src_on_path(V2_ROOT)
REPO_ROOT = V2_ROOT.parent
RAW = V2_ROOT / "data" / "raw"
INTERIM = V2_ROOT / "data" / "interim"
V3 = INTERIM / "v3"
REF = V2_ROOT / "data" / "reference"
ARTIFACTS = V2_ROOT / "artifacts"
FIGURES = V2_ROOT / "reports" / "figures" / "v3"
for d in (RAW, INTERIM, V3, REF, ARTIFACTS, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

SMOKE_TEST = True

def gate(*args, **kwargs):
    kwargs.setdefault("smoke_test", SMOKE_TEST)
    return _gate_impl(*args, **kwargs)

print("V2_ROOT =", V2_ROOT, "SMOKE_TEST =", SMOKE_TEST)


In [ ]:
from v3_payload import SCHEMA_VERSION, assert_payload_safe, copy_payloads_to_app, validate_cohort, validate_patient, v3_interim, glossary_allows_nll
from v3_smoke import assemble_v3, persist_smoke
import json

meta = ARTIFACTS / "poe_vae_meta.json"
encoder = json.loads(meta.read_text()).get("encoder", "jax_poe_vae") if meta.is_file() else "jax_poe_vae"
a1 = json.loads((V3 / "a1_meta.json").read_text()) if (V3 / "a1_meta.json").is_file() else {}
encoder = a1.get("encoder", encoder)

cohort, patients = assemble_v3(encoder=encoder)
# overlay gate files when present
if (V3 / "survival_stats.json").is_file():
    s = json.loads((V3 / "survival_stats.json").read_text())
    cohort["gates"]["a2"].update({"passed": s.get("passed"), "p_os": s.get("p_os"), "framing": s.get("framing")})
if (V3 / "a4_meta.json").is_file():
    a4 = json.loads((V3 / "a4_meta.json").read_text())
    cohort["gates"]["a4"]["passed"] = a4.get("passed", cohort["gates"]["a4"]["passed"])
    cohort["gates"]["a4"]["reversal_available"] = bool(a4.get("passed"))
if (V3 / "a1_meta.json").is_file():
    cohort["clustering_available"] = bool(a1.get("clustering_available", True))
    if not cohort["clustering_available"]:
        cohort["preregistered"]["k"] = None

assert validate_cohort(cohort) == []
for pid, payload in patients.items():
    if not glossary_allows_nll(encoder):
        payload.setdefault("limitations", []).append(
            "The displayed ellipse comes from a linear product-of-experts fallback; the VAE NLL gate does not apply."
        )
    assert validate_patient(payload, cohort) == []
    assert_payload_safe(payload, pid)
assert_payload_safe(cohort, "cohort")

dest = v3_interim(V2_ROOT)
(dest / "cohort_payload.json").write_text(json.dumps(cohort, indent=2))
for pid, payload in patients.items():
    (dest / f"payload_{pid}.json").write_text(json.dumps(payload, indent=2))
copy_payloads_to_app(cohort, patients, REPO_ROOT)
print("encoder", encoder, "n_patients", len(patients), "glossary_nll", glossary_allows_nll(encoder))


In [ ]:
from pathlib import Path
n_ok = 0
for path in [V3 / "cohort_payload.json", *V3.glob("payload_*.json")]:
    obj = json.loads(path.read_text())
    assert_safe(json.dumps(obj), context=str(path.name))
    n_ok += 1
gate("NB_A6", "payload_safety", float(n_ok), 1.0, note=f"assert_safe passed on {n_ok} payload files")
